# 10 - Kaggle readiness (run on account A and on account B)
Checks the GPUs, builds the environment, and runs a one-photo smoke test of every mesh model and appearance backend on GPU 0 and GPU 1. The result is `readiness.json` plus the built wheels, saved as notebook output so later sessions can attach them.

In [ ]:
REPO_URL = "https://github.com/saltypal/TrueFace3D.git"   # <- set before running
import os, subprocess, sys
REPO = "/kaggle/working/TrueFace3D"
if os.path.isdir(REPO + "/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
print("commit", subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
os.chdir(REPO); sys.path.insert(0, REPO)

In [ ]:
!nvidia-smi

In [ ]:
from src.kaggle import readiness
env = readiness.environment()
print(env['torch'])

Setup clones the pinned upstream repositories, links weights from the private dataset, and builds nvdiffrast / flame_solver / PyTorch3D for the T4 (sm_75). The first run takes a while; wheels land in `/kaggle/working/wheels`.

In [ ]:
from src.kaggle import setup
setup_report = setup.main([])

In [ ]:
OUT = '/kaggle/working/readiness'
rows = readiness.run_smoke(OUT, gpus=(0, 1))
table = readiness.readiness_table(rows)
from src.reporting import table_markdown
from IPython.display import Markdown
Markdown(table_markdown(table))

`ready` means the item passed on both GPUs. `not_run` means its code or weights are not available (see the detail column); that is expected for appearance methods whose weights are not in the dataset. `NOT READY` needs attention before the real run.

In [ ]:
readiness.save(OUT, env, rows)
print('saved', OUT)